# 🧠 Ryth — End-to-End Training (official entry point)

**One "Run All" notebook** that takes a raw coding corpus and produces a trained
Ryth model, using **only the existing Ryth library** (Corpus Engine, Tokenizer,
RDE, Model Core, Training Engine — none are modified).

```
Corpus Engine ─► Tokenizer ─► RDE (RDS) ─► Model ─► Training ─► Eval ─► Export
```

**Train any size** (30M / 125M / 350M / 1B) by changing **only the Configuration
cell** — everything else is generic and idempotent.

### Run on Kaggle
1. **Settings → Accelerator → GPU T4** (auto-detected; picks fp16 on T4, bf16 on Ampere+).
2. Get Ryth into the kernel: **Settings → Internet → On** (installs from GitHub),
   or attach the Ryth repo as a **Kaggle dataset** (auto-found under `/kaggle/input`).
3. **Run All.** Defaults run a fast, self-contained **smoke** build (synthetic data).
   For a real run: set `SMOKE = False` and point `RAW_DIR` / `GITHUB_SOURCES` at real code.

### Idempotent by design
Every stage detects existing outputs and **skips completed work**; training
**auto-resumes** from the latest checkpoint. Re-running "Run All" never redoes
expensive work unnecessarily.

## 1 · Environment Setup

In [ ]:
import os, sys, subprocess, importlib, platform

def on_kaggle():
    return os.path.isdir("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ

def have_ryth():
    try:
        for m in ("corpus", "tokenizer", "dataset", "model", "training"):
            importlib.import_module(m)
        return True
    except Exception:
        return False

RYTH_REPO = "https://github.com/RAJ-af/Ryth.git"
if not have_ryth():
    found = None
    if os.path.isdir("/kaggle/input"):
        for root, _dirs, files in os.walk("/kaggle/input"):
            if "pyproject.toml" in files and os.path.isdir(os.path.join(root, "corpus")):
                found = root; break
    if found:
        sys.path.insert(0, found); print("Using attached Ryth at", found)
    else:
        print("Installing Ryth from GitHub (needs Internet = On) …")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+" + RYTH_REPO],
                       check=False)

import torch

def pick_device_dtype(force=None):
    # T4 (Turing sm_75) has NO native bf16 -> fp16. Ampere+ (sm_80+) -> bf16.
    if not torch.cuda.is_available():
        return "cpu", (force or "fp32")
    major, _minor = torch.cuda.get_device_capability()
    return "cuda", (force or ("bf16" if major >= 8 else "fp16"))

KAGGLE = on_kaggle()
DEVICE, AUTO_DTYPE = pick_device_dtype()
print("=" * 62)
print("Ryth end-to-end   |   Kaggle:", KAGGLE)
print("Python:", platform.python_version(), "| torch:", torch.__version__)
print("Device:", DEVICE, "| auto dtype:", AUTO_DTYPE)
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(0), "| capability sm_%d%d" % cc)
print("Ryth importable:", have_ryth())
print("=" * 62)

## 2 · Configuration  ← change ONLY this cell

Everything configurable in one place. Flip `SMOKE = False` and fill in real
sources for a production run; switch `MODEL_PRESET` to scale 30M → 1B.

In [ ]:
SMOKE = True                 # fast, self-contained smoke build. False = real run.

# ---- data / corpus ----
RAW_DIR           = None                       # folder of real code (e.g. "/kaggle/input/my-code"); None = synthetic
GITHUB_SOURCES    = []                          # ["pallets/click", "psf/requests"]  (needs Internet)
HF_SOURCES        = []                          # ["code_search_net"]                (needs `datasets`)
LANGUAGES         = ["python", "javascript", "typescript", "go", "rust", "markdown"]
LICENSES          = ["MIT", "Apache-2.0", "BSD-2-Clause", "BSD-3-Clause", "ISC", "MPL-2.0"]
QUALITY_THRESHOLD = 0 if SMOKE else 40          # drop repos below this 0..100 score
TARGET_FILES      = None                        # cap kept files (None = all)

# ---- tokenizer / dataset ----
VOCAB_SIZE        = 1000 if SMOKE else 16000
SEQ_LEN           = 64   if SMOKE else 1024

# ---- model ----
MODEL_PRESET      = "ryth_30m"                  # ryth_30m | ryth_125m | ryth_350m | ryth_1b

# ---- training ----
MICRO_BATCH       = 4   if SMOKE else 16
GRAD_ACCUM        = 1   if SMOKE else 8
LR                = 3e-4
MAX_STEPS         = 12  if SMOKE else 5000
FORCE_DTYPE       = None                        # None = auto, or "bf16" | "fp16" | "fp32"

# ---- output / control ----
OUT_DIR           = "/kaggle/working/ryth_run" if os.path.isdir("/kaggle/working") else "ryth_run"
FORCE_REBUILD     = False                       # True = ignore caches, redo every stage

print("config ready | preset:", MODEL_PRESET, "| smoke:", SMOKE, "| out:", OUT_DIR)

### Derived paths + helpers

In [ ]:
import json, time, glob
from datetime import datetime, timezone

DTYPE      = FORCE_DTYPE or AUTO_DTYPE
CORPUS_DIR = os.path.join(OUT_DIR, "corpus")
TOK_DIR    = os.path.join(OUT_DIR, "tokenizer")
RDS_DIR    = os.path.join(OUT_DIR, "rds")
RUN_DIR    = os.path.join(OUT_DIR, "checkpoints")
for d in (OUT_DIR, CORPUS_DIR, TOK_DIR, RDS_DIR, RUN_DIR):
    os.makedirs(d, exist_ok=True)

TOK_JSON      = os.path.join(TOK_DIR, "tokenizer.json")
RECORDS_JSONL = os.path.join(CORPUS_DIR, "records.jsonl")
RDS_TRAIN     = os.path.join(RDS_DIR, "train")
RDS_VAL       = os.path.join(RDS_DIR, "validation")
FINAL_CKPT    = os.path.join(RUN_DIR, "final.pt")

def exists_nonempty(p):
    return os.path.exists(p) and (os.path.isdir(p) or os.path.getsize(p) > 0)

def stage_skip(name, path):
    if not FORCE_REBUILD and exists_nonempty(path):
        print(f"[skip] {name}: re=using {path}".replace("re=using", "reusing"))
        return True
    return False

def now_iso():
    return datetime.now(timezone.utc).isoformat()

print("dtype:", DTYPE)
print("out  :", OUT_DIR)

## 3 · Corpus Builder

Uses the **Corpus Engine** only: download → clean → dedup → filter → quality
score → language balance, plus task generation and HTML/JSON reports. When no
real sources are configured (or a fetch fails offline), a synthetic licensed
corpus is generated so the notebook always runs.

In [ ]:
from corpus import CorpusConfig, CorpusPipeline, Source, SourceList, build_task_dataset
from corpus.report import build_report, write_html_report, write_json_report
from corpus.metadata import RecordStore, write_repo_records
from corpus.exporters import export_tasks_jsonl

_T = [
    'def scale_{i}(x):\n    """Multiply x by {i}."""\n    return x * {i}\n',
    'def add_{i}(a, b):\n    total = a + b + {i}\n    return total\n',
    'class Box{i}:\n    def __init__(self, v):\n        self.v = v + {i}\n\n    def get(self):\n        return self.v\n',
    'def fib_{i}(n):\n    a, b = 0, {i}\n    for _ in range(n):\n        a, b = b, a + b\n    return a\n',
]
_MIT = "MIT License\n\nPermission is hereby granted, free of charge, to any person obtaining a copy of this software"

def make_synthetic(root, n_repos=16):
    for r in range(n_repos):
        d = os.path.join(root, f"repo_{r:02d}"); os.makedirs(d, exist_ok=True)
        open(os.path.join(d, "LICENSE"), "w").write(_MIT)
        open(os.path.join(d, "README.md"), "w").write(f"# repo_{r}\nA small utility library number {r}.\n")
        for k in range(5):
            j = r * 11 + k * 3 + 1
            open(os.path.join(d, f"mod_{k}.py"), "w").write(
                f'"""Module {k} of repo {r}."""\n\n'
                + _T[k % len(_T)].format(i=j) + "\n" + _T[(k + 1) % len(_T)].format(i=j + 7))
        open(os.path.join(d, "test_mod.py"), "w").write(
            f"from mod_0 import scale_{r*11+1}\n\n\ndef test_scale():\n    assert scale_{r*11+1}(2) == {(r*11+1)*2}\n")
    return root

def build_sources():
    sl = SourceList()
    for gh in GITHUB_SOURCES:
        sl.add(Source(id="gh:" + gh, kind="github", location=gh))
    for hf in HF_SOURCES:
        sl.add(Source(id="hf:" + hf, kind="huggingface", location=hf))
    raw = RAW_DIR
    if raw is None and not GITHUB_SOURCES and not HF_SOURCES:
        raw = os.path.join(OUT_DIR, "raw_synth")
        if not exists_nonempty(raw):
            make_synthetic(raw)
    if raw and os.path.isdir(raw):
        subs = [x for x in sorted(os.listdir(raw)) if os.path.isdir(os.path.join(raw, x))]
        if subs:
            for x in subs:
                sl.add(Source(id="local:" + x, kind="local", location=os.path.join(raw, x)))
        else:
            sl.add(Source(id="local:root", kind="local", location=raw))
    return sl

corpus_cfg = CorpusConfig(
    language_ratios={l: 1.0 for l in LANGUAGES},
    allowed_licenses=tuple(LICENSES),
    min_quality=QUALITY_THRESHOLD,
    split_ratios=({"train": 0.7, "validation": 0.15, "test": 0.15} if SMOKE
                  else {"train": 0.9, "validation": 0.05, "test": 0.05}),
)

if stage_skip("corpus", RECORDS_JSONL):
    records = list(RecordStore(RECORDS_JSONL).read())
    corpus_report = json.load(open(os.path.join(CORPUS_DIR, "report.json")))
else:
    pipe = CorpusPipeline(corpus_cfg)
    result = pipe.build(build_sources().enabled(), os.path.join(OUT_DIR, "stage"),
                        created_at=now_iso())
    if not result.records:                       # offline & remote-only -> synthetic
        print("[corpus] no files from configured sources; falling back to synthetic")
        raw = make_synthetic(os.path.join(OUT_DIR, "raw_synth"))
        sl = SourceList([Source(id="local:" + x, kind="local",
                                location=os.path.join(raw, x))
                         for x in sorted(os.listdir(raw))])
        result = pipe.build(sl.enabled(), os.path.join(OUT_DIR, "stage"), created_at=now_iso())
    records = result.records
    if TARGET_FILES:
        records = sorted(records, key=lambda r: r.hash)[:TARGET_FILES]
    RecordStore(RECORDS_JSONL).write(records, include_content=True)
    write_repo_records(os.path.join(CORPUS_DIR, "repos.jsonl"), result.repos)
    examples = build_task_dataset(records, corpus_cfg)
    export_tasks_jsonl(examples, os.path.join(CORPUS_DIR, "tasks.jsonl"))
    corpus_report = build_report(records, result.repos, examples=examples,
                                 drops=result.drops, config=corpus_cfg)
    write_json_report(corpus_report, os.path.join(CORPUS_DIR, "report.json"))
    write_html_report(corpus_report, os.path.join(CORPUS_DIR, "report.html"))

ds = corpus_report["dataset_size"]
print(f"[corpus] files={ds['files']} repos={ds['repositories']} MB={ds['megabytes']}")
print("[corpus] languages:", corpus_report["language_distribution"])
print("[corpus] splits   :", corpus_report["split_distribution"])
print("[corpus] tasks    :", corpus_report.get("task_distribution", {}))

## 4 · Tokenizer

Trains the scratch BPE tokenizer on the corpus text if `tokenizer.json` is
missing; otherwise loads the existing one.

In [ ]:
from tokenizer.train import train_tokenizer
from dataset import load_bpe_tokenizer

if not stage_skip("tokenizer", TOK_JSON):
    texts = [r.content for r in records if r.content]
    print(f"[tokenizer] training on {len(texts)} texts -> vocab≈{VOCAB_SIZE} …")
    train_tokenizer(texts, vocab_size=VOCAB_SIZE, out_dir=TOK_DIR, verbose=False)

tok = load_bpe_tokenizer(TOK_JSON)
print("[tokenizer] vocab_size =", tok.vocab_size)

## 5 · Dataset (RDE)

Builds Ryth RDS shards from the corpus **via the existing RDE** (one dataset per
split), only if not already built. Reuses them otherwise.

In [ ]:
from corpus.exporters import export_rds
from dataset import RDSDataset

if not stage_skip("rds", os.path.join(RDS_TRAIN, "manifest.json")):
    manifests = export_rds(records, RDS_DIR, tokenizer=tok, seq_len=SEQ_LEN)
    print("[rds] built splits:", list(manifests))

train_ds = RDSDataset(RDS_TRAIN)
has_val = os.path.exists(os.path.join(RDS_VAL, "manifest.json"))

# keep DataLoader (drop_last=True) fed: micro_batch must not exceed train chunks
eff_train = len(train_ds) - (0 if has_val else max(1, int(len(train_ds) * 0.02)))
if MICRO_BATCH > max(1, eff_train):
    MICRO_BATCH = max(1, eff_train)
    print(f"[rds] lowered MICRO_BATCH to {MICRO_BATCH} to fit {eff_train} train chunks")
print(f"[rds] train chunks={len(train_ds)} seq_len={train_ds.seq_len} val_split={has_val}")

## 6 · Model

In [ ]:
from model import RythConfig, RythForCausalLM

def make_model():
    mcfg = getattr(RythConfig, MODEL_PRESET)(vocab_size=tok.vocab_size)
    if SEQ_LEN > mcfg.max_seq_len:
        mcfg.max_seq_len = SEQ_LEN
    return RythForCausalLM(mcfg)

model = make_model()
n_params = model.num_params()
mc = model.config
print(f"[model] {MODEL_PRESET}: {n_params/1e6:.1f}M params | d_model={mc.d_model} "
      f"n_layers={mc.n_layers} n_heads={mc.n_heads} n_kv={mc.n_kv_heads} vocab={mc.vocab_size}")

## 7 · Training

The existing Training Engine: mixed precision, gradient accumulation/clipping,
checkpointing, **auto-resume**, JSON logging, validation + perplexity, and early
stopping. If a `final.pt` already exists the training stage is skipped; otherwise
it resumes from `latest.pt` when present.

In [ ]:
from training import TrainConfig, Trainer

train_cfg = TrainConfig(
    data_dir=RDS_TRAIN,
    val_data_dir=(RDS_VAL if has_val else None),
    model_preset=MODEL_PRESET, seq_len=SEQ_LEN,
    lr=LR, warmup_steps=max(1, MAX_STEPS // 10), max_steps=MAX_STEPS,
    micro_batch_size=MICRO_BATCH, grad_accum_steps=GRAD_ACCUM,
    dtype=DTYPE, out_dir=RUN_DIR,
    eval_every=max(2, MAX_STEPS // 4), eval_steps=5, log_every=max(1, MAX_STEPS // 10),
    save_every=max(2, MAX_STEPS // 2), early_stop_patience=(0 if SMOKE else 5),
    num_workers=(0 if SMOKE else 2), device=DEVICE, run_name="ryth-e2e",
    resume="latest")

_t0 = time.perf_counter()
if not FORCE_REBUILD and exists_nonempty(FINAL_CKPT):
    print("[train] final.pt exists -> training already complete, skipping")
    trained_best = None
else:
    trained_best = Trainer(train_cfg, model=model).train()
train_secs = time.perf_counter() - _t0
print(f"[train] {train_secs:.1f}s | best_val(from run)={trained_best}")

## 8 · Evaluation

Validation loss + perplexity (from the best checkpoint), training curves (from the
JSON log), and sample code generations.

In [ ]:
import math

metrics_path = os.path.join(RUN_DIR, "metrics.jsonl")
train_hist, eval_hist = [], []
if os.path.exists(metrics_path):
    for line in open(metrics_path):
        e = json.loads(line)
        (eval_hist if e.get("kind") == "eval" else train_hist).append(e)

best_val = None
for ck in ("best.pt", "final.pt", "latest.pt"):
    p = os.path.join(RUN_DIR, ck)
    if os.path.exists(p):
        best_val = torch.load(p, map_location="cpu", weights_only=False).get("best_val")
        break
best_ppl = math.exp(min(best_val, 20.0)) if best_val not in (None, float("inf")) else None
print(f"[eval] best_val_loss={best_val} | best_perplexity={best_ppl}")

# ---- training curve ----
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    if train_hist:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot([h["step"] for h in train_hist], [h["loss"] for h in train_hist],
                label="train loss")
        if eval_hist:
            ax.plot([h["step"] for h in eval_hist], [h["val_loss"] for h in eval_hist],
                    "o-", label="val loss")
        ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend()
        ax.set_title(f"Ryth {MODEL_PRESET} training curve")
        fig.tight_layout()
        fig.savefig(os.path.join(OUT_DIR, "training_curve.png"), dpi=110)
        try:
            plt.show()
        except Exception:
            pass
        print("[eval] saved training_curve.png")
except Exception as e:
    print("[eval] plot skipped:", e)
    for h in train_hist[-5:]:
        print("   step", h["step"], "loss", round(h["loss"], 4))

In [ ]:
from model import generate

ckpt = next((os.path.join(RUN_DIR, c) for c in ("best.pt", "final.pt", "latest.pt")
             if os.path.exists(os.path.join(RUN_DIR, c))), None)
gen_model = make_model()
if ckpt:
    state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    gen_model.load_state_dict(state["model"])
gen_model.to(DEVICE).eval()

for prompt in ("def add(a, b):\n", "class Stack:\n"):
    ids = tok.encode(prompt) or [getattr(tok, "bos_id", 0)]
    x = torch.tensor([ids[-SEQ_LEN:]], dtype=torch.long, device=DEVICE)
    out = generate(gen_model, x, max_new_tokens=48, temperature=0.8, top_k=40,
                   eos_id=getattr(tok, "eos_id", None))
    print("\n--- prompt:", repr(prompt), "---")
    print(tok.decode(out[0].tolist()))

## 9 · Export

All artifacts persist under `OUT_DIR` (on Kaggle: `/kaggle/working`, saved as
notebook output): tokenizer, RDS dataset, checkpoints (`best.pt`/`latest.pt`/
`final.pt`), `config.json`, and the corpus reports.

In [ ]:
run_config = {
    "smoke": SMOKE, "model_preset": MODEL_PRESET, "vocab_size": tok.vocab_size,
    "seq_len": SEQ_LEN, "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM,
    "lr": LR, "max_steps": MAX_STEPS, "dtype": DTYPE, "device": DEVICE,
    "languages": LANGUAGES, "licenses": LICENSES, "quality_threshold": QUALITY_THRESHOLD,
}
json.dump(run_config, open(os.path.join(OUT_DIR, "config.json"), "w"), indent=2)

artifacts = {
    "tokenizer": TOK_JSON,
    "rds": RDS_DIR,
    "checkpoints": sorted(os.path.basename(p) for p in glob.glob(os.path.join(RUN_DIR, "*.pt"))),
    "corpus_report_html": os.path.join(CORPUS_DIR, "report.html"),
    "corpus_report_json": os.path.join(CORPUS_DIR, "report.json"),
    "config": os.path.join(OUT_DIR, "config.json"),
    "training_curve": os.path.join(OUT_DIR, "training_curve.png"),
}
print(json.dumps(artifacts, indent=2))

## 10 · Resume / idempotency state

The stage cache below is what a re-run reuses. Delete `OUT_DIR` (or set
`FORCE_REBUILD = True`) to rebuild from scratch; delete only `checkpoints/final.pt`
to continue training from `latest.pt`.

In [ ]:
print("Stage cache (a re-run reuses these, resuming training from latest.pt):")
for name, path in [("corpus",     RECORDS_JSONL),
                   ("tokenizer",  TOK_JSON),
                   ("rds/train",  os.path.join(RDS_TRAIN, "manifest.json")),
                   ("checkpoint", os.path.join(RUN_DIR, "latest.pt")),
                   ("final",      FINAL_CKPT)]:
    print(f"  {'✓' if exists_nonempty(path) else '·'}  {name:12s} {path}")

## 11 · Final Summary

In [ ]:
summary = {
    "dataset_files":     ds["files"],
    "repositories":      ds["repositories"],
    "dataset_MB":        ds["megabytes"],
    "vocabulary_size":   tok.vocab_size,
    "model_preset":      MODEL_PRESET,
    "model_params_M":    round(n_params / 1e6, 2),
    "training_seconds":  round(train_secs, 1),
    "best_val_loss":     best_val,
    "best_perplexity":   round(best_ppl, 4) if best_ppl else None,
    "output_dir":        OUT_DIR,
}
json.dump(summary, open(os.path.join(OUT_DIR, "summary.json"), "w"), indent=2)

print("=" * 62)
print(f"  RYTH END-TO-END SUMMARY  ({MODEL_PRESET})")
print("=" * 62)
for k, v in summary.items():
    print(f"  {k:18s}: {v}")
print("=" * 62)
print("Scale up: set SMOKE=False, point RAW_DIR/GITHUB_SOURCES at real code,")
print("switch MODEL_PRESET to ryth_125m / ryth_350m / ryth_1b, then Run All.")